# v3.0.0 — Calculator API Guide (Backwards Compatible)

- **Pre-v3**: The library was structured around calculator classes that returned aggregated data as a Pydantic model, integrating with the UKRDC API.
- **v3.0.0**: The library now supports additional functionality, such as returning patient lists.
- The syntax for calling calculators remains broadly the same.

In [1]:
import json
import datetime as dt
from ukrdc_stats.calculators.krt import KRTStatsCalculator 
from dotenv import dotenv_values
from ukrdc_stats.utils.database import get_sessionmaker

config = dotenv_values("../.env")
KEYPATH = config.get("UKRDC_STATS_KEYPATH")
SERVER = "ukrdc_live"

# note that this requires an open ssh tunnel from either port 6100 or 6200 to 5432 of the relevent db server
# substitute choice of db connection method
ukrdc_session =  get_sessionmaker(SERVER, keypath=KEYPATH)()

facility = "RNJ00"
end = dt.datetime.now()
start = end - dt.timedelta(days = 90)


calculator = KRTStatsCalculator(session=ukrdc_session, facility=facility, from_time=start, to_time=end)
#calculator = UnitLevelKRTStats(session=session, facility=facility, from_time=start, to_time=end)
output = calculator.extract_stats()


formatted_output = json.dumps(
    json.loads(
        output.all.json()
    ), 
    indent=4
)

print(formatted_output)

{
    "incident_krt_modality": {
        "metadata": {
            "title": "Incident KRT Modality",
            "summary": "Modality breakdown of incident KRT patients",
            "description": "\n# Incident Kidney Replacement Patients\n\n## Definition\nA patient starting kidney replacement therapy (KRT) - defined as haemodialysis, peritoneal dialysis, \nor kidney transplant - for the first time during the selected time period.\n\n## Inclusion Criteria\n1. First treatment starts within the selected dates. This is defined as the first treatment preceded by a gap of greater than 90 days without KRT treatment.\n2. Either:\n   - Known kidney disease history (planned start)\n   - No prior history (unplanned/\"crash\" start) AND survives >90 days\n3. Treatment continues for at least 90 days OR patient:\n   - Has planned start and dies within 90 days\n   - Transfers to another unit\n4. Patient is modality is assigned to the first \n\n## Timeline Example\n```\nKey:\nX = Treatment\n- = No T

C:\Users\philip.main\AppData\Local\Temp\ipykernel_23428\291963715.py:27: PydanticDeprecatedSince20: The `json` method is deprecated; use `model_dump_json` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  output.all.json()


# Continued
- Basic modality headcount for incident and prevalent KRT

# Discontinued
- Demographics stats for all patients sent to the UKRDC
- HD session frequency and incident initial access — will return once useful definitions are agreed

# New
- Demographics can be calculated for each calculator
- CKD calculator now yields the same statistics as the KRT calculator
- Care planning

In [ ]:
# The following plots are largely unchanged (the class has been slightly modified)

import plotly.express as px

prevalent_modalities = output.all.prevalent_krt_modality

all_patients = px.pie(
    names=prevalent_modalities.data.x,
    values=prevalent_modalities.data.y,
    hole=0.3,
    title=f"{prevalent_modalities.metadata.title} ({facility})",
)
all_patients.add_annotation(
    text=f"{prevalent_modalities.metadata.population_size}",
    x=0.5,
    y=0.5,
    font_size=16,
    showarrow=False,
)
all_patients.show()

incident_modalities = output.all.incident_krt_modality

all_patients = px.pie(
    names=incident_modalities.data.x,
    values=incident_modalities.data.y,
    hole=0.3,
    title=f"{incident_modalities.metadata.title} ({facility})",
)
all_patients.add_annotation(
    text=f"{incident_modalities.metadata.population_size}",
    x=0.5,
    y=0.5,
    font_size=16,
    showarrow=False,
)
all_patients.show()

In [ ]:
# Gender (sex) pie charts — incident and prevalent KRT

for stat in (
    output.all.incident_krt_sex,
    output.all.prevalent_krt_sex,
):
    fig = px.pie(
        names=stat.data.x,
        values=stat.data.y,
        hole=0.3,
        title=f"{stat.metadata.title} ({facility})",
    )
    fig.add_annotation(
        text=f"{stat.metadata.population_size}",
        x=0.5,
        y=0.5,
        font_size=16,
        showarrow=False,
    )
    fig.show()

# Ethnicity pie charts — incident and prevalent KRT

for stat in (
    output.all.incident_krt_ethnicity,
    output.all.prevalent_krt_ethnicity,
):
    fig = px.pie(
        names=stat.data.x,
        values=stat.data.y,
        hole=0.3,
        title=f"{stat.metadata.title} ({facility})",
    )
    fig.add_annotation(
        text=f"{stat.metadata.population_size}",
        x=0.5,
        y=0.5,
        font_size=16,
        showarrow=False,
    )
    fig.show()

# Age bar charts — incident and prevalent KRT

for stat in (
    output.all.incident_krt_age,
    output.all.prevalent_krt_age,
):
    fig = px.bar(
        x=stat.data.x,
        y=stat.data.y,
        title=f"{stat.metadata.title} ({facility})",
        labels={"x": stat.metadata.title, "y": "Count"},
    )
    fig.show()

# Prevalent CKD Calculator

- `PrevalentCKDCalculator` produces the same set of stats as the KRT calculator.
- The cohort is calculated for a single prevalence point rather than a time window.
- Same `extract_stats()` syntax, returning stats for the whole centre plus a per-satellite-unit drilldown.

In [ ]:
from ukrdc_stats.calculators.ckd import PrevalentCKDCalculator

facility = 'RP5'

ckd_calculator = PrevalentCKDCalculator(
    session=ukrdc_session,
    facility=facility,
    prevalence_point=end,
)
ckd_output = ckd_calculator.extract_stats()

print(f"Satellite units: {', '.join(ckd_output.units.keys())}")
print(f"Total population: {ckd_output.all.metadata.population}")

Satellite units: RP5
Total population: 12


In [ ]:
# same styling as the KRT plots: donut with the cohort population in the hole

for stat in (
    ckd_output.all.prevalent_ckd_age,
    ckd_output.all.prevalent_ckd_ethnicity,
    ckd_output.all.prevalent_ckd_sex,
):
    fig = px.pie(
        names=stat.data.x,
        values=stat.data.y,
        hole=0.3,
        title=f"{stat.metadata.title} ({facility})",
    )
    fig.add_annotation(
        text=f"{stat.metadata.population_size}",
        x=0.5,
        y=0.5,
        font_size=16,
        showarrow=False,
    )
    fig.show()

# New Measures

- Experimenting with care choice (assessment) data.
- This data is now linked to all cohorts.

In [ ]:
# Careplanning pie charts — incident KRT, prevalent KRT, and prevalent CKD

# KRT careplanning (reuses the KRT calculator output from earlier)
for stat in (
    output.all.incident_krt_careplanning,
    output.all.prevalent_krt_careplanning,
):
    fig = px.pie(
        names=stat.data.x,
        values=stat.data.y,
        hole=0.3,
        title=f"{stat.metadata.title} ({facility})",
    )
    fig.add_annotation(
        text=f"{stat.metadata.population_size}",
        x=0.5,
        y=0.5,
        font_size=16,
        showarrow=False,
    )
    fig.show()

# CKD careplanning (reuses the CKD calculator output from earlier)
ckd_careplanning = ckd_output.all.prevalent_ckd_careplanning
fig = px.pie(
    names=ckd_careplanning.data.x,
    values=ckd_careplanning.data.y,
    hole=0.3,
    title=f"{ckd_careplanning.metadata.title} ({ckd_calculator.facility})",
)
fig.add_annotation(
    text=f"{ckd_careplanning.metadata.population_size}",
    x=0.5,
    y=0.5,
    font_size=16,
    showarrow=False,
)
fig.show()